In [3]:
"""
使用预训练的 Flan-T5-large without Fine-Tuning
"""

import json
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==================== 1. load model and tokenizer ====================
model_name = "google/flan-t5-large"  # 可选: t5-base, t5-large, flan-t5-xl 等

print(f"loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# device GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()  # evaluation mode, no training
print(f"The model has been loaded into: {device}")

# ==================== 2. read test dataset ====================
test_file_path = "/root/autodl-fs/test_ner_clean.jsonl"

print(f"reading test dataset: {test_file_path}")
test_data = []
with open(test_file_path, "r", encoding="utf-8") as f:
    for line in f:
        test_data.append(json.loads(line.strip()))

print(f"test has {len(test_data)} samples")

正在加载模型: google/flan-t5-large


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/flan-t5-large/resolve/main/tokenizer_config.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc12454da90>: Failed to establish a new connection: [Errno 101] Network is unreachable'))"), '(Request ID: cbb7f602-cb3d-4578-afdf-694e67182512)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-large/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/flan-t5-large/resolve/main/tokenizer_config.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc12454f2f0>: Failed to establish a new connection: [Errno 101] Network is unreachable'))"), '(Request ID: a23ada73-b764-40fd-a09f-104d80ed999d)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-large/resolve/main/toke

模型已加载到: cuda
正在读取测试集: /root/autodl-fs/test_ner_clean.jsonl
测试集共 6658 条数据


In [18]:
# ==================== 3. Generation ====================
def generate_summary(article_text, max_input_length=1024, max_output_length=256):

    # 
    # "summarize: " as prefix
    input_text = f"summarize: {article_text}"
    
    # tokenization
    inputs = tokenizer(
        input_text, 
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # 生成摘要
    # 参考 README: "decode" 命令中的 generation 参数
    with torch.no_grad():  # 推理时不需要梯度
        outputs = model.generate(
            inputs["input_ids"],
            max_length=max_output_length,
            min_length=30,
            num_beams=4,          # 束搜索，提高质量
            early_stopping=True,
            no_repeat_ngram_size=3  # 避免重复
        )
    
    # 解码
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary


# 生成所有摘要（可以先测试几条）
print("\n开始生成摘要...")
results = []

# 可选：先测试前 n 条确认效果
test_mode = True  # 设为 False 跑全部数据
sample_limit = 1000 if test_mode else len(test_data)

for i, item in enumerate(tqdm(test_data[:sample_limit], desc="生成进度")):
    # 获取文章内容（根据你的 JSON 结构调整 key）
    # 常见字段名: "article", "text", "source", "input"
    article = item.get("article") or item.get("text") or item.get("source")
    
    if article is None:
        print(f"警告: 第 {i} 条数据没有找到文章字段，跳过")
        continue
    
    # 生成摘要
    summary = generate_summary(article)
    
    # 保存结果
    results.append({
        "index": i,
        "original_article": article[:200] + "..." if len(article) > 200 else article,
        "generated_summary": summary,
        # 如果有原文摘要，可以保留用于后续评估
        "reference_abstract": item.get("abstract") or item.get("target")
    })

# ==================== 4. 保存结果 ====================
output_file = "generated_summaries.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n生成完成！共生成 {len(results)} 条摘要")
print(f"结果已保存到: {output_file}")

# 打印一个示例
if results:
    print("\n===== 示例摘要 =====")
    print(f"原文片段: {results[0]['original_article']}")
    print(f"生成摘要: {results[0]['generated_summary']}")


开始生成摘要...


生成进度: 100%|██████████| 1000/1000 [14:08<00:00,  1.18it/s]


生成完成！共生成 1000 条摘要
结果已保存到: generated_summaries.json

===== 示例摘要 =====
原文片段: anxiety affects quality of life in those living with parkinson 's disease ( pd ) more so than overall cognitive status , motor deficits , apathy , and depression [ 13 ] . although anxiety and depressi...
生成摘要: pd patients with anxiety and ten patients without anxiety ) were included in the study . the study included patients with and without anxiety in parkinson 's disease , regardless of symptom laterality .


In [19]:
"""
计算 ROUGE-1、ROUGE-2、ROUGE-L 指标
注意：由于服务器网络限制，无法连接 Hugging Face，BERTScore 部分已跳过。
"""

import json
from evaluate import load

print("加载评估指标...")
rouge_metric = load("rouge")

# 读取之前生成的摘要结果
results_file = "generated_summaries.json"

with open(results_file, "r", encoding="utf-8") as f:
    results = json.load(f)

# 提取预测摘要和参考答案
predictions = [item["generated_summary"] for item in results]
references = [item["reference_abstract"] for item in results]

print(f"共加载 {len(predictions)} 条数据，开始计算 ROUGE 指标...")

# 计算 ROUGE 指标
rouge_result = rouge_metric.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

# 只输出 ROUGE 结果
print("\n" + "="*50)
print("📊 评估结果 (ROUGE)")
print("="*50)
print(f"ROUGE-1:     {rouge_result['rouge1']:.4f}")
print(f"ROUGE-2:     {rouge_result['rouge2']:.4f}")
print(f"ROUGE-L:     {rouge_result['rougeL']:.4f}")
print("="*50)
print("注: BERTScore 因网络连接问题未能计算。")

# 保存结果
metrics = {
    "rouge1": round(rouge_result["rouge1"], 4),
    "rouge2": round(rouge_result["rouge2"], 4),
    "rougeL": round(rouge_result["rougeL"], 4),
    "bertscore_f1": None,
    "note": "BERTScore skipped due to network connection error to huggingface.co",
    "num_samples": len(predictions)
}

with open("evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"\nROUGE 指标已保存到 evaluation_metrics.json")

加载评估指标...


共加载 1000 条数据，开始计算 ROUGE 指标...

📊 评估结果 (ROUGE)
ROUGE-1:     0.1628
ROUGE-2:     0.0727
ROUGE-L:     0.1218
注: BERTScore 因网络连接问题未能计算。

ROUGE 指标已保存到 evaluation_metrics.json
